# Kalanos · Audit Cube to Bowl before training

**10 episodes · ~2m 30s of recorded data · 30 FPS · 1 episode to review**

**What Kalanos found:** Episode 000008 had lower gripper-camera sharpness than that camera’s usual baseline. The other 9 episodes passed the recorded checks.

**Dataset:** [Cube to Bowl](https://huggingface.co/datasets/triton7777/cube_to_bowl_so100) · SO-100 pick-and-place · [Pinned LeRobot revision](https://huggingface.co/datasets/triton7777/cube_to_bowl_so100/commit/9f490c4e624034eb5c6e961e65767abb9ab7c391)

**Try it:** Choose **Runtime → Run all** to audit the dataset, inspect the episode evidence, and preview the JSON. No API key or GPU required. The notebook does not publish your report.

[Kalanos](https://kalanos.ai) · [Open-source grader](https://github.com/KalanosAI/kalanos)

## 01 · Load Cube to Bowl
Install the pinned grader and download the public dataset snapshot.

In [ ]:
# @title Install Kalanos (CPU)
import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

from IPython.display import HTML, display


WORK = Path(tempfile.mkdtemp(prefix="kalanos-colab-"))
VENV = WORK / "venv"
OUTPUT = WORK / "results"
OUTPUT.mkdir()
# Colab's Python 3.13 lacks ensurepip, so `python -m venv` cannot seed pip.
# virtualenv bundles its own pip and works on Colab and local Python alike.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "-q", str(VENV)], check=True)
PYTHON = VENV / "bin" / "python"
KALANOS = VENV / "bin" / "kalanos"
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "kalanos[numeric,video]==0.10.0",
        "huggingface-hub==1.31.0",
    ],
    check=True,
)
freeze = subprocess.run(
    [str(PYTHON), "-m", "pip", "freeze"], capture_output=True, text=True
)
lock = freeze.stdout if freeze.returncode == 0 else "# pip freeze unavailable\n"
(OUTPUT / "requirements-resolved.txt").write_text(lock)
print("Kalanos 0.10.0 is ready.")

In [ ]:
# @title Download the pinned Cube to Bowl snapshot
DATASET_ID = "triton7777/cube_to_bowl_so100"
DATASET_REVISION = "9f490c4e624034eb5c6e961e65767abb9ab7c391"
MAX_BYTES = 200 * 1024 * 1024

if DATASET_REVISION and not re.fullmatch(r"[a-f0-9]{40}", DATASET_REVISION):
    raise ValueError(
        "Use an exact Hugging Face commit SHA, or leave blank to resolve once."
    )

download_script = r"""import json, pathlib, sys
from huggingface_hub import HfApi, snapshot_download
repo, revision, destination, max_bytes = sys.argv[1:]
api = HfApi(token=False)
info = api.dataset_info(repo, revision=revision or "main", files_metadata=True)
files = info.siblings
if not files or any(f.size is None for f in files):
    raise RuntimeError("Cannot verify repository size; no download performed.")
total = sum(f.size for f in files)
if total > int(max_bytes):
    raise RuntimeError(f"Repository is {total} bytes, above download limit.")
path = snapshot_download(repo_id=repo, repo_type="dataset", revision=info.sha,
                         local_dir=destination, token=False, max_workers=2)
print(json.dumps({"dataset_id":repo, "revision":info.sha,
                  "repository_bytes":total, "local_path":path}))
"""
env = dict(os.environ, HF_HUB_DISABLE_TELEMETRY="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
raw = subprocess.check_output(
    [
        str(PYTHON),
        "-c",
        download_script,
        DATASET_ID,
        DATASET_REVISION,
        str(WORK / "download"),
        str(MAX_BYTES),
    ],
    text=True,
    env=env,
)
provenance = json.loads(raw.strip().splitlines()[-1])
roots = [
    p.parent.parent for p in Path(provenance["local_path"]).rglob("meta/info.json")
]
if len(roots) != 1:
    raise RuntimeError(
        f"Expected one unpacked LeRobot root, found {len(roots)}. "
        "Inspect repository layout."
    )
DATASET = roots[0]
info = json.loads((DATASET / "meta/info.json").read_text())
if not list((DATASET / "data").rglob("*.parquet")):
    raise RuntimeError(
        "No LeRobot Parquet data found; do not grade metadata as recordings."
    )
provenance.update(
    {
        "declared_format": info.get("codebase_version"),
        "declared_episodes": info.get("total_episodes"),
        "grader_version": "0.10.0",
        "python_version": sys.version,
    }
)
(OUTPUT / "provenance.json").write_text(json.dumps(provenance, indent=2))
print(
    f"{DATASET_ID} loaded · {info['total_episodes']} declared episodes · "
    f"{info['fps']} FPS · {info['codebase_version']}"
)

## 02 · Run the audit

Review or blocked decisions are results, not notebook errors. This run uses the full tier and vision-imitation profile. The [published Hub assessment](https://hub.kalanos.ai/d/@kalanos/hf/triton7777/cube_to_bowl_so100) is an independent run using an earlier grader release. Kalanos runs once per export format; the summary below comes from the JSON run. The HTML is a separate assessment of the same inputs and settings.

In [ ]:
# @title Generate HTML and JSON reports
grader_env = dict(os.environ, NO_COLOR="1")
grader_env.pop("KALANOS_API_KEY", None)


def grade_report(suffix):
    report = OUTPUT / f"audit.{suffix}"
    command = [
        str(KALANOS),
        "grade",
        str(DATASET),
        "--profile",
        "vision-imitation-v1",
        "--tier",
        "full",
        "--report",
        str(report),
        "--fail-on",
        "blocked,review,unknown",
    ]
    result = subprocess.run(command, env=grader_env, capture_output=True, text=True)
    (OUTPUT / f"grader-{suffix}.log").write_text(result.stdout + "\n" + result.stderr)
    if result.returncode not in (0, 1) or not report.is_file():
        raise RuntimeError(
            f"Grading failed (exit {result.returncode}). " + result.stderr[-3000:]
        )
    print(
        "Gate:", "passed" if result.returncode == 0 else "failed; review the assessment"
    )
    return report


JSON_REPORT = grade_report("json")
HTML_REPORT = grade_report("html")

## 03 · Inspect the findings

Start with the decision counts, then expand a finding in the HTML report to see the recorded evidence. Compare what was checked with what was unavailable. A descriptive score is not a readiness decision or a prediction of training success.

In [ ]:
# @title Show the Kalanos results and review evidence
from html import escape


report = json.loads(JSON_REPORT.read_text())
counts = report.get("eligibility_counts") or {}
findings = report.get("findings") or []


def safe(value):
    return escape(str(value if value is not None else "Not reported"))


def number(value):
    return f"{value:.4g}" if isinstance(value, (int, float)) else safe(value)


total, review = safe(counts.get("total")), safe(counts.get("review"))
ready = (report.get("readiness") or {}).get("score")
readiness = (
    f"{ready:.1f} / 100" if isinstance(ready, (int, float)) else "Not established"
)
decisions = " · ".join(
    f"{label} {safe(counts.get(key))}"
    for label, key in [
        ("Pass", "pass_count"),
        ("Review", "review"),
        ("Blocked", "blocked"),
        ("Unknown", "unknown"),
    ]
)
cards = []
for finding in findings:
    if not isinstance(finding, dict):
        continue
    evidence = finding.get("evidence") or {}
    metric = finding.get("metric_id") or finding.get("metric")
    episode = finding.get("episode_id") or finding.get("episode")
    if isinstance(episode, str):
        episode = episode.split("::")[-1]
    camera = (
        finding.get("source_field") or evidence.get("camera") or finding.get("stream")
    )
    label = (
        "Gripper-camera sharpness outlier"
        if metric == "vision.blur_vs_camera" and "gripper" in str(camera)
        else safe(metric)
    )
    baseline = evidence.get("camera_median")
    baseline_line = (
        f"<li>Same-camera median: {number(baseline)} across {safe(evidence.get('n_episodes'))} episodes</li>"
        if baseline is not None
        else ""
    )
    cards.append(
        f'<details style="margin-top:14px;padding:15px;border:1px solid #d9e1ed;border-radius:10px;background:#f8fafc;color:#0f1d36">'
        f'<summary style="cursor:pointer;font-weight:700">{label} · {safe(episode)}</summary>'
        f"<ul><li>Camera: {safe(camera)}</li><li>Measured: {number(finding.get('value'))} {safe(finding.get('unit'))}</li>"
        f"{baseline_line}<li>Decision: {safe(finding.get('consequence'))}</li></ul>"
        f'<p style="margin-bottom:0">Metric: <code>{safe(metric)}</code></p></details>'
    )

evidence_html = "".join(cards) or "<p>No findings recorded in this run.</p>"
display(
    HTML(f"""
<section style="font:15px system-ui;background:#0f1d36;color:#f8fafc;border-radius:14px;padding:22px;margin:8px 0 14px">
  <div style="letter-spacing:3px;color:#87dfd6;font-weight:700">KALANOS</div>
  <h2 style="color:#f8fafc;margin:8px 0">{total} episodes · {review} to review</h2>
  <p style="color:#e2e8f0">Cube to Bowl · triton7777/cube_to_bowl_so100</p>
  <p style="color:#f8fafc;margin-bottom:8px">{decisions}</p>
  <small style="color:#c9d8ee">Technical readiness: {readiness}</small>
</section>
{evidence_html}
<p style="font:13px system-ui;color:#52647c">The complete HTML report and original audit JSON are available in the export step below.</p>
""")
)

## 04 · Preview the audit JSON

Open **Preview audit.json** to see how the result and episode finding are recorded. Download the full JSON for an experiment or CI check, or export the HTML report and provenance separately.

In [ ]:
# @title Preview audit.json and export reports
import json

import ipywidgets as widgets
from google.colab import files
from IPython.display import HTML, display


def preview_audit(path):
    audit = json.loads(path.read_text(encoding="utf-8"))
    count_keys = ("total", "pass_count", "review", "blocked", "unknown")
    finding_keys = ("episode_id", "metric_id", "severity", "value", "unit")
    preview = {
        "schema_version": audit.get("schema_version"),
        "eligibility_counts": {
            key: audit.get("eligibility_counts", {}).get(key) for key in count_keys
        },
        "findings": [
            {key: finding.get(key) for key in finding_keys}
            for finding in audit.get("findings", [])[:3]
        ],
    }
    return escape(json.dumps(preview, indent=2))


display(
    HTML(f"""
<details style="padding:12px;border:1px solid #b9cbd4;border-radius:8px">
  <summary style="cursor:pointer;font-weight:700">Preview audit.json</summary>
  <p style="margin:8px 0">Selected fields from the original audit JSON. No download needed.</p>
  <pre style="font-size:14px;white-space:pre-wrap;margin:4px 0">{preview_audit(JSON_REPORT)}</pre>
</details>
""")
)


def download_button(label, path):
    button = widgets.Button(description=label, layout=widgets.Layout(width="auto"))
    button.on_click(lambda _: files.download(str(path)))
    return button


display(
    widgets.HBox(
        [
            download_button("Download audit.json", JSON_REPORT),
            download_button("Download audit.html", HTML_REPORT),
            download_button("Download provenance.json", OUTPUT / "provenance.json"),
        ],
        layout=widgets.Layout(flex_flow="row wrap", gap="8px"),
    )
)

**Continue with Kalanos:** [Browse dataset reports](https://hub.kalanos.ai) · [Use the grader locally](https://github.com/KalanosAI/kalanos) · [Attach an audit to a training run](https://github.com/KalanosAI/kalanos-tracking)